**Project 2: What drives house prices in Ames? A SQL analysis**

I uploaded the Ames dataset into SQLite database and i used SQL to answer five property questions a property investor might ask

**SETUP**

In [3]:
import sqlite3
import pandas as pd

df = pd.read_csv("train.csv")
conn = sqlite3.connect("housing.db")
df.to_sql("houses", conn, if_exists = "replace", index=False)

def q(sql):
  return pd.read_sql(sql, conn)

conn.execute("DROP TABLE IF EXISTS hood_stats")
conn.execute("""
CREATE TABLE hood_stats AS
SELECT Neighborhood, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY Neighborhood
""")
conn.commit()

q("SELECT COUNT(*) AS number_of_houses FROM houses")


,number_of_houses
0,1460


**MOST AND LEAST EXPENSIVE NEIGHBORHOODS**

In [4]:
q("""
SELECT Neighborhood, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY Neighborhood
HAVING COUNT(*) > 50
ORDER BY avg_price DESC
""")

,Neighborhood,houses,avg_price
0,NridgHt,77,316271.0
1,Somerst,86,225380.0
2,Crawfor,51,210625.0
3,CollgCr,150,197966.0
4,Gilbert,79,192855.0
5,NWAmes,73,189050.0
6,SawyerW,59,186556.0
7,NAmes,225,145847.0
8,Sawyer,74,136793.0
9,OldTown,113,128225.0


*Among the 12 neighborhoods with more than 50 sales, NridgHt is the most expensive (average 316,271) and BrkSide the cheapest (average 124,834), a gap of about 2.5 times.*

**PRICE PER SQUARE FOOT**

In [7]:
q("""
SELECT Neighborhood, ROUND(AVG(SalePrice * 1.0 / GrLivArea), 1) AS price_per_sqft
FROM houses
GROUP BY Neighborhood
ORDER BY price_per_sqft DESC
LIMIT 5
"""
)

,Neighborhood,price_per_sqft
0,StoneBr,165.4
1,NridgHt,164.6
2,Veenker,155.0
3,Somerst,141.3
4,Timber,139.8


*StoneBr and NridgHt have the highest average price per square foot (about 165), against a typical figure of about 120.*

**FEATURES THAT GO WITH HIGHER PRICES**

In [8]:
q("""
SELECT OverallQual, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY OverallQual
ORDER BY OverallQual
""")

,OverallQual,houses,avg_price
0,1,2,50150.0
1,2,3,51770.0
2,3,20,87474.0
3,4,116,108421.0
4,5,397,133523.0
5,6,374,161603.0
6,7,319,207716.0
7,8,168,274736.0
8,9,43,367513.0
9,10,18,438588.0


In [9]:
q("""
SELECT GarageCars, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY GarageCars
ORDER BY GarageCars
""")

,GarageCars,houses,avg_price
0,0,81,103317.0
1,1,369,128117.0
2,2,824,183852.0
3,3,181,309636.0
4,4,5,192656.0


In [10]:
q("""
SELECT CentralAir, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY CentralAir
ORDER BY CentralAir
""")

,CentralAir,houses,avg_price
0,N,95,105264.0
1,Y,1365,186187.0


*Average price rises at every step of overall quality, from about 50,150 at quality 1 to 438,588 at quality 10. Houses with a 3-car garage average about 309,636, roughly 3 times the no-garage average of 103,317, and houses with central air average 186,187 against 105,264 without it. These features also go with house size, age, and quality, so none of them explains price alone.*

**HOUSES THAT SOLD FAR ABOVE THEIR MARKET**

In [12]:
q("""
SELECT h.Neighborhood, h.SalePrice, s.avg_price,
       h.SalePrice - s.avg_price AS above_avg
FROM houses AS h
JOIN hood_stats AS s ON h.Neighborhood = s.Neighborhood
ORDER BY above_avg DESC
LIMIT 10
""")

,Neighborhood,SalePrice,avg_price,above_avg
0,NoRidge,755000,335295.0,419705.0
1,NoRidge,745000,335295.0,409705.0
2,OldTown,475000,128225.0,346775.0
3,NridgHt,611657,316271.0,295386.0
4,NoRidge,625000,335295.0,289705.0
5,NridgHt,582933,316271.0,266662.0
6,StoneBr,556581,310499.0,246082.0
7,NridgHt,555000,316271.0,238729.0
8,StoneBr,538000,310499.0,227501.0
9,CollgCr,424870,197966.0,226904.0


In [13]:
q("""
SELECT COUNT(*) AS above_their_hood_avg
FROM houses AS h
JOIN hood_stats AS s ON h.Neighborhood = s.Neighborhood
WHERE h.SalePrice > s.avg_price
""")

,above_their_hood_avg
0,636


*The biggest outperformer is a NoRidge house that sold 419,705 above its neighborhood average. 636 of the 1,460 houses (about 44%) sold above their neighborhood average, fewer than half because a few very expensive houses pull each average up.*

*the OldTown house in third place (475,000, built in 1892, quality 10) beat its neighborhood average by 346,775, so a high-quality house can outperform a cheap neighborhood.*

**WHERE HIGH QUALITY HOUSES CLUSTER**

In [14]:
q("""
SELECT h.Neighborhood, COUNT(*) AS strong_houses
FROM houses AS h
JOIN hood_stats AS s ON h.Neighborhood = s.Neighborhood
WHERE h.OverallQual >= 8 AND h.SalePrice > s.avg_price
GROUP BY h.Neighborhood
ORDER BY strong_houses DESC
LIMIT 5
""")

,Neighborhood,strong_houses
0,NridgHt,38
1,Somerst,28
2,CollgCr,25
3,Timber,14
4,NoRidge,13


*NridgHt has the most high-quality houses (quality 8 or higher) that sold above their own neighborhood average (38), followed by Somerst (28) and CollgCr (25). NoRidge has only 13, because it has few houses and a high bar to beat.*

**SUMMARY**

*Price in Ames depends on quality, location, and size together, and no single factor explains it. Quality has the clearest pattern: average price rises at every step of the score, from about 50,150 to 438,588. Location matters too: among the larger neighborhoods, NridgHt averages about 2.5 times BrkSide, and StoneBr and NridgHt charge the most per square foot (about 165). The 1892 OldTown house with quality 10 sold 346,775 above its neighborhood average, so quality can outweigh location. Next, I would test these links with a model, to see how much each factor adds once the others are held constant.*